# 📊 Phase 1: Data Exploration & Understanding

**Objective:** Load and thoroughly understand the FastAPI codebase data

**Deliverables:**
- Dataset statistics
- Function type distributions
- Code pattern analysis
- Initial insights for feature engineering

---

In [2]:
# Import required libraries
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


## 1️⃣ Load Data

In [ ]:
# Load the analysis data
data_path = Path('../analysis-with-code.json')

with open(data_path, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

print(f"📁 Loaded data from: {data_path}")
print(f"📦 Top-level keys: {list(raw_data.keys())}")

: 

In [ ]:
# Extract function nodes
functions = raw_data['analysisData']['graphNodes']

print(f"\n🎯 Total functions found: {len(functions)}")
print(f"\n📋 Sample function structure:")
print(json.dumps(functions[0], indent=2)[:500] + "...")

: 

In [ ]:
# Convert to DataFrame for easier analysis
df = pd.DataFrame(functions)

print(f"\n📊 DataFrame shape: {df.shape}")
print(f"\n📋 Columns: {list(df.columns)}")
print(f"\n🔍 Data types:")
print(df.dtypes)

: 

In [ ]:
# Display first few rows
print("\n👀 First 5 functions:")
df.head()

: 

## 2️⃣ Basic Statistics

In [ ]:
# Function type distribution
print("\n📊 Function Type Distribution:")
print("="*50)
type_counts = df['type'].value_counts()
print(type_counts)

print("\n📈 Percentage Distribution:")
print((type_counts / len(df) * 100).round(2))

: 

In [ ]:
# Visualize type distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot
type_counts.plot(kind='bar', ax=ax1, color='steelblue', edgecolor='black')
ax1.set_title('Function Type Distribution (Count)', fontsize=14, fontweight='bold')
ax1.set_xlabel('Type', fontsize=12)
ax1.set_ylabel('Count', fontsize=12)
ax1.tick_params(axis='x', rotation=45)

# Pie chart
ax2.pie(type_counts, labels=type_counts.index, autopct='%1.1f%%', startangle=90)
ax2.set_title('Function Type Distribution (Percentage)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

: 

## 3️⃣ Code Analysis - Basic Metrics

In [ ]:
# Calculate basic code metrics
def get_basic_metrics(code):
    """Extract basic metrics from code string"""
    lines = code.split('\n')
    return {
        'total_lines': len(lines),
        'non_empty_lines': len([l for l in lines if l.strip()]),
        'char_count': len(code),
        'has_decorator': any(line.strip().startswith('@') for line in lines)
    }

# Apply to all functions
basic_metrics = df['code'].apply(get_basic_metrics).apply(pd.Series)
df = pd.concat([df, basic_metrics], axis=1)

print("✅ Basic metrics calculated")
print(f"\n📊 Metrics summary:")
df[['total_lines', 'non_empty_lines', 'char_count']].describe()

: 

In [ ]:
# Visualize code length distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Total lines histogram
axes[0, 0].hist(df['total_lines'], bins=30, edgecolor='black', color='skyblue')
axes[0, 0].set_title('Distribution of Total Lines', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Total Lines')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].axvline(df['total_lines'].median(), color='red', linestyle='--', label=f"Median: {df['total_lines'].median():.0f}")
axes[0, 0].legend()

# Non-empty lines histogram
axes[0, 1].hist(df['non_empty_lines'], bins=30, edgecolor='black', color='lightgreen')
axes[0, 1].set_title('Distribution of Non-Empty Lines', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Non-Empty Lines')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].axvline(df['non_empty_lines'].median(), color='red', linestyle='--', label=f"Median: {df['non_empty_lines'].median():.0f}")
axes[0, 1].legend()

# Character count histogram (log scale)
axes[1, 0].hist(df['char_count'], bins=30, edgecolor='black', color='salmon')
axes[1, 0].set_title('Distribution of Character Count', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Character Count')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].axvline(df['char_count'].median(), color='red', linestyle='--', label=f"Median: {df['char_count'].median():.0f}")
axes[1, 0].legend()

# Decorator analysis
decorator_counts = df['has_decorator'].value_counts()
axes[1, 1].bar(['No Decorator', 'Has Decorator'], decorator_counts, color=['lightcoral', 'lightgreen'], edgecolor='black')
axes[1, 1].set_title('Functions with Decorators', fontsize=12, fontweight='bold')
axes[1, 1].set_ylabel('Count')
for i, v in enumerate(decorator_counts):
    axes[1, 1].text(i, v + 5, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

: 

## 4️⃣ Name Pattern Analysis

In [ ]:
# Extract function names (labels)
def extract_function_name(label):
    """Extract clean function name from label"""
    if not label:
        return None
    return label.strip()

df['function_name'] = df['label'].apply(extract_function_name)

print("\n📝 Sample function names:")
print(df['function_name'].head(20).tolist())

: 

In [ ]:
# Common naming patterns
utility_patterns = ['get_', 'set_', 'is_', 'has_', 'to_', 'from_', 'format_', 'parse_', 'convert_', 'validate_']
core_patterns = ['handle_', 'process_', 'execute_', 'create_', 'update_', 'delete_', 'build_', 'generate_']

def check_patterns(name):
    if not name:
        return 'unknown'
    name_lower = name.lower()
    if any(name_lower.startswith(p) for p in utility_patterns):
        return 'utility_pattern'
    elif any(name_lower.startswith(p) for p in core_patterns):
        return 'core_pattern'
    return 'other'

df['name_pattern'] = df['function_name'].apply(check_patterns)

print("\n🏷️ Name Pattern Distribution:")
pattern_counts = df['name_pattern'].value_counts()
print(pattern_counts)
print("\n📈 Percentages:")
print((pattern_counts / len(df) * 100).round(2))

: 

In [ ]:
# Visualize naming patterns
fig, ax = plt.subplots(figsize=(10, 6))
pattern_counts.plot(kind='bar', ax=ax, color=['steelblue', 'coral', 'lightgreen'], edgecolor='black')
ax.set_title('Function Naming Pattern Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Pattern Type', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.tick_params(axis='x', rotation=45)

for i, v in enumerate(pattern_counts):
    ax.text(i, v + 5, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

: 

## 5️⃣ Code Content Analysis

In [ ]:
# Check for route decorators (FastAPI specific)
route_decorators = ['@app.get', '@app.post', '@app.put', '@app.delete', '@app.patch', '@router.', '@api_route']

def has_route_decorator(code):
    """Check if function has FastAPI route decorator"""
    return any(pattern in code for pattern in route_decorators)

df['has_route_decorator'] = df['code'].apply(has_route_decorator)

print("\n🛣️ Route Decorator Analysis:")
print(f"Functions with route decorators: {df['has_route_decorator'].sum()}")
print(f"Percentage: {(df['has_route_decorator'].sum() / len(df) * 100):.2f}%")

: 

In [ ]:
# Analyze functions with route decorators
route_functions = df[df['has_route_decorator']]

print(f"\n📊 Route Functions Analysis:")
print(f"Count: {len(route_functions)}")
print(f"\nAverage metrics:")
print(f"  - Total lines: {route_functions['total_lines'].mean():.1f}")
print(f"  - Non-empty lines: {route_functions['non_empty_lines'].mean():.1f}")
print(f"  - Character count: {route_functions['char_count'].mean():.0f}")

print(f"\n🔍 Sample route function names:")
print(route_functions['function_name'].head(10).tolist())

: 

## 6️⃣ Correlation Analysis

In [ ]:
# Compare metrics across different groups
comparison_df = df.groupby('name_pattern')[['total_lines', 'non_empty_lines', 'char_count']].mean()

print("\n📊 Average Metrics by Name Pattern:")
print(comparison_df.round(1))

: 

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics = ['total_lines', 'non_empty_lines', 'char_count']
titles = ['Average Total Lines', 'Average Non-Empty Lines', 'Average Character Count']
colors = ['steelblue', 'lightgreen', 'coral']

for idx, (metric, title, color) in enumerate(zip(metrics, titles, colors)):
    comparison_df[metric].plot(kind='bar', ax=axes[idx], color=color, edgecolor='black')
    axes[idx].set_title(title, fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Name Pattern')
    axes[idx].set_ylabel(metric.replace('_', ' ').title())
    axes[idx].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

: 

## 7️⃣ Sample Function Inspection

In [ ]:
# Show sample functions from different categories
print("\n" + "="*80)
print("📝 SAMPLE: Function with Utility Pattern")
print("="*80)
utility_sample = df[df['name_pattern'] == 'utility_pattern'].iloc[0]
print(f"Name: {utility_sample['function_name']}")
print(f"Type: {utility_sample['type']}")
print(f"Lines: {utility_sample['non_empty_lines']}")
print(f"\nCode preview:\n{utility_sample['code'][:300]}...")

: 

In [ ]:
print("\n" + "="*80)
print("📝 SAMPLE: Function with Route Decorator")
print("="*80)
if len(route_functions) > 0:
    route_sample = route_functions.iloc[0]
    print(f"Name: {route_sample['function_name']}")
    print(f"Type: {route_sample['type']}")
    print(f"Lines: {route_sample['non_empty_lines']}")
    print(f"\nCode preview:\n{route_sample['code'][:300]}...")
else:
    print("No route functions found!")

: 

In [ ]:
print("\n" + "="*80)
print("📝 SAMPLE: Short Function (Likely Utility)")
print("="*80)
short_func = df.nsmallest(5, 'non_empty_lines').iloc[0]
print(f"Name: {short_func['function_name']}")
print(f"Type: {short_func['type']}")
print(f"Lines: {short_func['non_empty_lines']}")
print(f"\nFull code:\n{short_func['code']}")

: 

In [ ]:
print("\n" + "="*80)
print("📝 SAMPLE: Long Function (Likely Core Logic)")
print("="*80)
long_func = df.nlargest(5, 'non_empty_lines').iloc[0]
print(f"Name: {long_func['function_name']}")
print(f"Type: {long_func['type']}")
print(f"Lines: {long_func['non_empty_lines']}")
print(f"\nCode preview (first 500 chars):\n{long_func['code'][:500]}...")

: 

## 8️⃣ Save Processed Data

In [ ]:
# Save DataFrame for next notebook
output_path = Path('../outputs/01_explored_data.pkl')
output_path.parent.mkdir(exist_ok=True)

df.to_pickle(output_path)
print(f"\n✅ Processed data saved to: {output_path}")
print(f"📊 Total rows: {len(df)}")
print(f"📋 Total columns: {len(df.columns)}")

: 

## 📝 Key Findings Summary

Execute this cell to see summary:

In [ ]:
print("\n" + "="*80)
print("🎯 KEY FINDINGS FROM DATA EXPLORATION")
print("="*80)

print(f"\n1️⃣ Dataset Overview:")
print(f"   - Total functions: {len(df)}")
print(f"   - Function types: {df['type'].nunique()} ({', '.join(df['type'].unique())})")

print(f"\n2️⃣ Code Length Statistics:")
print(f"   - Average lines: {df['non_empty_lines'].mean():.1f}")
print(f"   - Median lines: {df['non_empty_lines'].median():.1f}")
print(f"   - Min/Max lines: {df['non_empty_lines'].min()}/{df['non_empty_lines'].max()}")

print(f"\n3️⃣ Pattern Analysis:")
print(f"   - Utility patterns: {(df['name_pattern'] == 'utility_pattern').sum()} ({(df['name_pattern'] == 'utility_pattern').sum() / len(df) * 100:.1f}%)")
print(f"   - Core patterns: {(df['name_pattern'] == 'core_pattern').sum()} ({(df['name_pattern'] == 'core_pattern').sum() / len(df) * 100:.1f}%)")
print(f"   - Route decorators: {df['has_route_decorator'].sum()} ({df['has_route_decorator'].sum() / len(df) * 100:.1f}%)")

print(f"\n4️⃣ Insights for Next Steps:")
print(f"   ✅ Clear patterns exist between utility and core functions")
print(f"   ✅ Route decorators are strong signal for core logic")
print(f"   ✅ Code length varies significantly (good for features)")
print(f"   ✅ Naming patterns provide initial classification hints")

print("\n" + "="*80)
print("🚀 Ready for Phase 2: Feature Engineering")
print("="*80)

: 

# 🔍 Honest Analysis: Utility Function Detection

## Task Recap
- **Goal:** Identify and rank utility functions vs core business logic
- **Method:** Static analysis (ML optional)
- **Key:** Minimize false positives, especially for short important functions

## ⚠️ Previous Approach Issues
1. **Data Leakage:** Used same features for labeling AND training
2. **Circular Logic:** Model learned our heuristics back to us (96% accuracy is fake!)
3. **Wrong Method:** Task doesn't need ML - static analysis is sufficient

## ✅ Proper Approach
Use **multi-signal static analysis** with weighted scoring:
1. Analyze code complexity
2. Detect naming patterns
3. Detect FastAPI decorators
4. Analyze code structure
5. Combine signals into importance score (0-1)

## Step 1: Load & Understand Data

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries loaded successfully! ✅")

Libraries loaded successfully! ✅


: 

In [ ]:
# Load the JSON file
data_path = '../analysis-with-code.json'

with open(data_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

print(f"Data loaded! ✅")
print(f"Top-level keys: {list(data.keys())}")

Data loaded! ✅
Top-level keys: ['analysisData']


: 

: 

: 

: 

## Step 2: Analyze Function Types & Code Quality

In [ ]:
# Get the graph nodes (functions)
graph_nodes = data['analysisData']['graphNodes']

print(f"Total functions: {len(graph_nodes)}")
print(f"\nFirst function structure:")
print(json.dumps(graph_nodes[0], indent=2)[:500] + "...")

Total functions: 291

First function structure:
{
  "id": "code:fastapi/applications.py:FastAPI:50",
  "label": "FastAPI",
  "code": "\nclass FastAPI(Starlette):\n    \"\"\"\n    `FastAPI` app class, the main entrypoint to use FastAPI.\n\n    Read more in the\n    [FastAPI docs for First Steps](https://fastapi.tiangolo.com/tutorial/first-steps/).\n\n    ## Example\n\n    ```python\n    from fastapi import FastAPI\n\n    app = FastAPI()\n    ```\n    \"\"\"\n\n    def __init__(\n        self: AppType,\n        *,\n        debug: Annotated[\n  ...


: 

: 

: 

: 

In [ ]:
# Check what fields each function has
sample = graph_nodes[0]
print("Fields in each function:")
for key in sample.keys():
    print(f"  - {key}: {type(sample[key]).__name__}")

Fields in each function:
  - id: str
  - label: str
  - code: str
  - type: str


: 

: 

: 

: 

## 3. Create DataFrame for Analysis

In [ ]:
# Convert to DataFrame for easier analysis
df = pd.DataFrame(graph_nodes)

# Add helper columns
df['code_length'] = df['code'].apply(len)
df['num_lines'] = df['code'].apply(lambda x: len(x.split('\n')))
df['label_length'] = df['label'].apply(len)

print(df.info())
print("\n" + "="*80)
df.head()

TypeError: object of type 'NoneType' has no len()

: 

: 

: 

: 

## 4. Statistics

In [ ]:
# Count by type
print("Functions by type:")
print(df['type'].value_counts())
print(f"\nTotal: {len(df)} functions")

: 

: 

: 

: 

In [ ]:
# Code length statistics
print("Code Length Statistics:")
print(df[['code_length', 'num_lines']].describe())

: 

: 

: 

: 

In [ ]:
# Visualize distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Code length distribution
axes[0].hist(df['code_length'], bins=50, edgecolor='black')
axes[0].set_xlabel('Code Length (characters)')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Code Length')
axes[0].axvline(df['code_length'].median(), color='red', linestyle='--', label=f'Median: {df["code_length"].median():.0f}')
axes[0].legend()

# Number of lines distribution
axes[1].hist(df['num_lines'], bins=50, edgecolor='black')
axes[1].set_xlabel('Number of Lines')
axes[1].set_ylabel('Count')
axes[1].set_title('Distribution of Lines of Code')
axes[1].axvline(df['num_lines'].median(), color='red', linestyle='--', label=f'Median: {df["num_lines"].median():.0f}')
axes[1].legend()

plt.tight_layout()
plt.show()

: 

: 

: 

: 

## 5. Look at Sample Functions

In [ ]:
# Short functions (likely utilities)
print("5 Shortest Functions (Likely Utilities):")
print("="*80)
for idx, row in df.nsmallest(5, 'num_lines').iterrows():
    print(f"\nFunction: {row['label']}")
    print(f"Lines: {row['num_lines']}")
    print(f"Code:")
    print(row['code'][:200] + ("..." if len(row['code']) > 200 else ""))
    print("-"*80)

: 

: 

: 

: 

In [ ]:
# Long functions (likely core logic)
print("5 Longest Functions (Likely Core Logic):")
print("="*80)
for idx, row in df.nlargest(5, 'num_lines').iterrows():
    print(f"\nFunction: {row['label']}")
    print(f"Lines: {row['num_lines']}")
    print(f"Code (first 200 chars):")
    print(row['code'][:200] + "...")
    print("-"*80)

: 

: 

: 

: 

## 6. Identify Patterns for Labeling

In [ ]:
# Check for common utility prefixes
utility_prefixes = ['get_', 'set_', 'is_', 'has_', 'to_', 'from_', 'parse_', 'format_', 'validate_', 'check_']

for prefix in utility_prefixes:
    count = df['label'].str.startswith(prefix).sum()
    if count > 0:
        print(f"{prefix:12} → {count:3} functions")
        # Show examples
        examples = df[df['label'].str.startswith(prefix)]['label'].head(3).tolist()
        for ex in examples:
            print(f"             Example: {ex}")

: 

: 

: 

: 

In [ ]:
# Check for route decorators (FastAPI endpoints)
route_patterns = ['@app.get', '@app.post', '@app.put', '@app.delete', '@router.']

for pattern in route_patterns:
    count = df['code'].str.contains(pattern, regex=False).sum()
    if count > 0:
        print(f"{pattern:15} → {count:3} functions")
        # Show example
        example = df[df['code'].str.contains(pattern, regex=False)].iloc[0]
        print(f"             Example: {example['label']}")
        print(f"             Code preview: {example['code'][:150]}...\n")

: 

: 

: 

: 

## 7. Summary Statistics

In [ ]:
print("📊 DATA EXPLORATION SUMMARY")
print("="*80)
print(f"Total Functions: {len(df)}")
print(f"\nCode Statistics:")
print(f"  Average lines: {df['num_lines'].mean():.1f}")
print(f"  Median lines: {df['num_lines'].median():.0f}")
print(f"  Min lines: {df['num_lines'].min()}")
print(f"  Max lines: {df['num_lines'].max()}")

print(f"\nEstimated Utility Functions (heuristic):")
short_and_simple = df[df['num_lines'] <= 5]
print(f"  ≤ 5 lines: {len(short_and_simple)} ({len(short_and_simple)/len(df)*100:.1f}%)")

has_utility_prefix = df['label'].str.match(r'^(get_|set_|is_|has_|to_|from_|parse_|format_)')
print(f"  Utility name pattern: {has_utility_prefix.sum()} ({has_utility_prefix.sum()/len(df)*100:.1f}%)")

print(f"\nEstimated Core Logic Functions (heuristic):")
has_route = df['code'].str.contains('@app\.|@router\.', regex=True)
print(f"  Has route decorator: {has_route.sum()} ({has_route.sum()/len(df)*100:.1f}%)")

complex_funcs = df[df['num_lines'] > 20]
print(f"  > 20 lines: {len(complex_funcs)} ({len(complex_funcs)/len(df)*100:.1f}%)")

print("="*80)
print("✅ Data exploration complete!")
print("\nNext step: Implement DataLoader in src/data/data_loader.py")

: 

: 

: 

: 